# Run LSAS For One Dataset

This notebook runs the simplified LSAS analysis flow for one chosen generated CSV dataset.

It calls `run_dataset(...)` directly, rather than launching the command-line script. The script loads one CSV dataset, splits it into graph blocks, runs LSAS for each selected Markov blanket discovery method, and writes one matching `_lsas.csv` file.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent, cwd.parent.parent]

for candidate in candidate_roots:
    analysis_dir = candidate / "NeurIPS_2026" / "analysis"
    if analysis_dir.exists():
        project_root = candidate
        break
else:
    if (cwd / "run_analysis.py").exists():
        analysis_dir = cwd
        project_root = cwd.parent.parent
    else:
        raise RuntimeError("Could not find NeurIPS_2026/analysis")

if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import run_analysis

neurips_dir = project_root / "NeurIPS_2026"
data_root = neurips_dir / "data" / "generated"
results_dir = neurips_dir / "analysis" / "results"

print(f"project_root = {project_root}")
print(f"data_root    = {data_root}")
print(f"results_dir  = {results_dir}")

project_root = C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome
data_root    = C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome\NeurIPS_2026\data\generated
results_dir  = C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome\NeurIPS_2026\analysis\results


## Choose Dataset

In [ ]:
available_datasets = run_analysis.discover_datasets(data_root)

print(f"Found {len(available_datasets)} datasets")
for index, path in enumerate(available_datasets):
    print(f"{index}: {path.relative_to(data_root).as_posix()}")

In [ ]:
# Option A: choose by index from the printed list above.
dataset_index = 0
source_file = available_datasets[dataset_index]

# Option B: choose explicitly by relative path.
# source_file = data_root / "gaussian" / "output_cyclic_linear_graphs2_n1000_nodes8_edges12_t0_y1_20260429-1718.csv"

print(source_file.relative_to(data_root).as_posix())

## Configure LSAS Run

In [ ]:
mb_methods = ["TC", "IAMB", "Fast_IAMB", "HITON_MB"]
raw_max_k = "auto"
limit_graphs = None
overwrite = True

# Examples:
# mb_methods = ["TC"]
# raw_max_k = "4"
# limit_graphs = 1

## Run Dataset

In [ ]:
run_analysis.setup_logging()
run_analysis.load_runtime_dependencies()

output_path = run_analysis.run_dataset(
    source_file=source_file,
    data_root=data_root,
    results_dir=results_dir,
    mb_methods=mb_methods,
    raw_max_k=raw_max_k,
    limit_graphs=limit_graphs,
    overwrite=overwrite,
)

output_path

## Summarise Metrics And Plot


In [2]:
import summarise_lsas_metrics
import plot_lsas_metrics
from collections import defaultdict

results_metric_dir = analysis_dir / "results_metric"
figures_dir = analysis_dir / "figures"
metrics_summary_path = results_metric_dir / "metrics_summary.csv"
output_path = Path(r"C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome\NeurIPS_2026\analysis\results\test_error.csv")

summarise_lsas_metrics.load_runtime_dependencies()
processed_rows = summarise_lsas_metrics.processed_rows_for_file(output_path)

grouped = defaultdict(list)
for row in processed_rows:
    grouped[str(row.get("mb_method", ""))].append(row)

summary_rows = [
    summarise_lsas_metrics.summarize_group(rows, output_path, mb_method)
    for mb_method, rows in sorted(grouped.items())
]

summarise_lsas_metrics.write_csv(
    metrics_summary_path,
    summary_rows,
    summarise_lsas_metrics.SUMMARY_COLUMNS,
)

plot_rows = plot_lsas_metrics.read_summary(metrics_summary_path)
figure_paths = plot_lsas_metrics.create_plots(
    plot_rows,
    figures_dir,
    formats=["png"],
    lines=False,
)

print(f"Wrote summary metrics: {metrics_summary_path}")
print(f"Wrote {len(figure_paths)} figure file(s) to {figures_dir}")

metrics_summary_path, figure_paths[:5]


Wrote summary metrics: C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome\NeurIPS_2026\analysis\results_metric\metrics_summary.csv
Wrote 9 figure file(s) to C:\Users\gijsv\OneDrive - Universiteit Leiden\phd\code\gutmicrobiome\gutmicrobiome\NeurIPS_2026\analysis\figures


(WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/NeurIPS_2026/analysis/results_metric/metrics_summary.csv'),
 [WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/NeurIPS_2026/analysis/figures/re/acyclic_small_graphs_pretrue_re_nodes15.png'),
  WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/NeurIPS_2026/analysis/figures/ntest/acyclic_small_graphs_pretrue_ntest_nodes15.png'),
  WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/NeurIPS_2026/analysis/figures/cpu_time/acyclic_small_graphs_pretrue_cpu_time_nodes15.png'),
  WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/NeurIPS_2026/analysis/figures/wall_time/acyclic_small_graphs_pretrue_wall_time_nodes15.png'),
  WindowsPath('C:/Users/gijsv/OneDrive - Universiteit Leiden/phd/code/gutmicrobiome/gutmicrobiome/N

## Inspect Output

In [ ]:
import pandas as pd

lsas_results = pd.read_csv(output_path)
lsas_results[["visible_node_columns"]]